In [ ]:
import os
import sys
import yaml
import pandas as pd

current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, project_root)

from functions.feature_selection import FeatureSelectionOrchestrator
from utils.plots import Pearson_correlation, Bar_plot
from Titanic.src.features.feature_eng import PreprocessingOrchestrator

In [5]:
with open(os.path.join("../config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)
        
with open(os.path.join( "../config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)  

# Load dataset

In [6]:
X_train = pd.read_parquet(
        os.path.join(
            config['init_path'],
            config['data']['processed'],
            "train_features.parquet")
    )
y_train = X_train[['Survived']]
    
preprocessor = PreprocessingOrchestrator(
        numerical_con=config_pipe['features']['num_con'], 
        numerical_dis=config_pipe['features']['num_dis'], 
        categorical_var=config_pipe['features']['cat_var'])
    
pipe = preprocessor.apply("preprocessing")
        
X_train_trans = pipe.fit_transform(X_train)

In [7]:
X_train_trans

,numerical_pipe_con__Age,numerical_pipe_con__Fare,numerical_pipe_dis__IsAlone,numerical_pipe_dis__FamilySize,categorical_pipe__Pclass,categorical_pipe__Sex,categorical_pipe__Embarked,categorical_pipe__Title,categorical_pipe__Cabin_1p
0,22.0,7.2500,0,2,3,male,S,Mr,missing
1,38.0,71.2833,0,2,1,female,C,Mrs,C
2,26.0,7.9250,1,1,3,female,S,Miss,missing
3,35.0,53.1000,0,2,1,female,S,Mrs,C
4,35.0,8.0500,1,1,3,male,S,Mr,missing
...,...,...,...,...,...,...,...,...,...
886,27.0,13.0000,1,1,2,male,S,Rare,missing
887,19.0,30.0000,1,1,1,female,S,Miss,B
888,28.0,23.4500,0,4,3,female,S,Miss,missing
889,26.0,30.0000,1,1,1,male,C,Mr,C


# Univariate Feat Selection

In [8]:
feature_selection = FeatureSelectionOrchestrator()

In [9]:
QuiSquare = feature_selection.apply(
        "QuiSquare", 
        X_train_trans.filter(like='categorical'), 
        y_train)

Feature: categorical_pipe__Pclass
Feature: categorical_pipe__Sex
Feature: categorical_pipe__Embarked
Feature: categorical_pipe__Title
Feature: categorical_pipe__Cabin_1p


In [10]:
QuiSquare

categorical_pipe__Title       3.957861e-61
categorical_pipe__Sex         1.197357e-58
categorical_pipe__Pclass      4.549252e-23
categorical_pipe__Cabin_1p    6.326020e-18
categorical_pipe__Embarked    1.618719e-06
Name: QuiSquare, dtype: float64

In [11]:
Anova = feature_selection.apply(
        "Anova",
        X_train_trans.filter(like='numerical'),
        y_train)
    
mi = feature_selection.apply(
        "MutualInformationClassif", 
        X_train_trans.filter(like='numerical'), 
        y_train)
    
corr = feature_selection.apply(
        "PearsonCorrelation", 
        X_train_trans.filter(like='numerical'), 
        y_train)

c:\Users\gustavo\anaconda3\envs\mf_tf\lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\gustavo\anaconda3\envs\mf_tf\lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [12]:
Anova

numerical_pipe_con__Fare          6.120189e-15
numerical_pipe_dis__IsAlone       9.009490e-10
numerical_pipe_con__Age           5.276069e-02
numerical_pipe_dis__FamilySize    6.198911e-01
Name: Anova, dtype: float64

In [13]:
mi

numerical_pipe_con__Fare          0.143095
numerical_pipe_dis__FamilySize    0.066743
numerical_pipe_con__Age           0.010414
numerical_pipe_dis__IsAlone       0.010096
Name: mutual information, dtype: float64

In [14]:
corr

,numerical_pipe_con__Age,numerical_pipe_con__Fare,numerical_pipe_dis__IsAlone,numerical_pipe_dis__FamilySize
numerical_pipe_con__Age,1.000000,0.096688,0.171647,-0.245619
numerical_pipe_con__Fare,0.096688,1.000000,-0.271832,0.217138
numerical_pipe_dis__IsAlone,0.171647,-0.271832,1.000000,-0.690922
numerical_pipe_dis__FamilySize,-0.245619,0.217138,-0.690922,1.000000
